# 03 Model Evaluation and Predictions
## Evaluating Model Performance and Making Future Predictions

This notebook uses the trained model to evaluate performance and make predictions.

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.data_loader import download_stock_data
from src.preprocessing import DataPreprocessor, create_dataset, reshape_for_lstm
from src.model import LSTMModel

print("Libraries imported successfully!")

## Step 1: Load Trained Model

In [ ]:
ticker = 'AAPL'
model_path = f'../models/saved_models/{ticker}_lstm_model.h5'
config_path = f'../models/saved_models/{ticker}_config.json'

print(f"Loading model and configuration...")

# Load configuration
with open(config_path, 'r') as f:
    config = json.load(f)

print(f"Model Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load model
model = LSTMModel(**config)
model.load_model(model_path)

time_steps = config['time_steps']

## Step 2: Prepare Recent Data for Evaluation

In [ ]:
# Load recent data
print(f"\nDownloading recent data...")
end_date = datetime.now()
start_date = end_date - timedelta(days=120)

data = download_stock_data(
    ticker,
    start_date.strftime('%Y-%m-%d'),
    end_date.strftime('%Y-%m-%d')
)

# Preprocess
preprocessor = DataPreprocessor()
data = preprocessor.handle_missing_values(data)
scaled_data = preprocessor.scale_data(data[['Close']])

print(f"Data prepared: {len(data)} records")

## Step 3: Evaluate on Recent Data

In [ ]:
# Create sequences
X, y = create_dataset(scaled_data, time_step=time_steps)
X_reshaped = reshape_for_lstm(X)

# Make predictions
print(f"\nMaking predictions on recent data...")
y_pred = model.predict(X_reshaped)

# Inverse transform
y_actual = preprocessor.inverse_transform(y.reshape(-1, 1))
y_pred_actual = preprocessor.inverse_transform(y_pred)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_actual, y_pred_actual))
mae = mean_absolute_error(y_actual, y_pred_actual)
r2 = r2_score(y_actual, y_pred_actual)

# Directional accuracy
direction_actual = np.diff(y_actual.flatten()) > 0
direction_pred = np.diff(y_pred_actual.flatten()) > 0
directional_accuracy = np.mean(direction_actual == direction_pred) * 100

print(f"\n{'='*70}")
print("RECENT PERFORMANCE METRICS")
print(f"{'='*70}")
print(f"RMSE: ${rmse:.2f}")
print(f"MAE: ${mae:.2f}")
print(f"R²: {r2:.4f}")
print(f"Directional Accuracy: {directional_accuracy:.2f}%")
print(f"{'='*70}")

## Step 4: Visualize Recent Predictions

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(y_actual, label='Actual Price', linewidth=2.5, alpha=0.8, color='blue')
plt.plot(y_pred_actual, label='Predicted Price', linewidth=2.5, alpha=0.8, color='red')
plt.title(f'{ticker} - Recent Model Performance', fontsize=14, fontweight='bold')
plt.xlabel('Time Period', fontsize=12)
plt.ylabel('Stock Price ($)', fontsize=12)
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5: Make Future Predictions

In [ ]:
# Parameters for future prediction
days_ahead = 30

print(f"\nGenerating {days_ahead}-day price forecast...")

# Get latest sequence
last_sequence = scaled_data[-time_steps:].reshape(1, time_steps, 1)

# Make predictions
predictions = []
current_sequence = last_sequence.copy()

for day in range(days_ahead):
    # Predict next price
    next_pred = model.predict(current_sequence, verbose=0)
    predictions.append(next_pred[0, 0])
    
    # Update sequence for next prediction
    current_sequence = np.append(current_sequence[:, 1:, :], 
                                next_pred.reshape(1, 1, 1), axis=1)

# Inverse transform predictions
predictions = np.array(predictions).reshape(-1, 1)
predictions_actual = preprocessor.inverse_transform(predictions)

# Get last actual price
last_actual_price = preprocessor.inverse_transform(scaled_data[-1:].reshape(1, 1))

print(f"Forecast completed!")

## Step 6: Display Future Predictions

In [ ]:
# Create results dataframe
future_dates = [end_date + timedelta(days=i) for i in range(1, days_ahead + 1)]

results_df = pd.DataFrame({
    'Date': future_dates,
    'Predicted_Price': predictions_actual.flatten(),
    'Price_Change': np.diff(
        np.concatenate([[last_actual_price[0, 0]], predictions_actual.flatten()])
    ),
})

results_df['Price_Change_Pct'] = (results_df['Price_Change'] / 
                                  results_df['Predicted_Price'].shift(1) * 100)

print(f"\n{'='*70}")
print(f"{days_ahead}-DAY PRICE FORECAST")
print(f"{'='*70}")
print(f"\nCurrent Price ({ticker}): ${last_actual_price[0, 0]:.2f}")
print(f"Forecast Period: {days_ahead} days")
print(f"Start Date: {end_date.date()}")
print(f"End Date: {future_dates[-1].date()}")
print(f"\n{results_df.to_string(index=False)}")

## Step 7: Forecast Analysis

In [ ]:
first_pred = predictions_actual.flatten()[0]
last_pred = predictions_actual.flatten()[-1]
avg_pred = predictions_actual.flatten().mean()
change_pct = ((last_pred - last_actual_price[0, 0]) / last_actual_price[0, 0]) * 100

print(f"\n{'='*70}")
print("FORECAST ANALYSIS")
print(f"{'='*70}")
print(f"\nStarting Prediction (Day 1): ${first_pred:.2f}")
print(f"Ending Prediction (Day {days_ahead}): ${last_pred:.2f}")
print(f"Average Predicted Price: ${avg_pred:.2f}")
print(f"Minimum Predicted Price: ${predictions_actual.min():.2f}")
print(f"Maximum Predicted Price: ${predictions_actual.max():.2f}")
print(f"\nTotal Expected Change: ${last_pred - last_actual_price[0, 0]:.2f}")
print(f"Total Expected Change %: {change_pct:.2f}%")

if change_pct > 2:
    trend = "🔺 BULLISH"
elif change_pct < -2:
    trend = "🔻 BEARISH"
else:
    trend = "➡️  NEUTRAL"

print(f"\nPredicted Trend: {trend} ({change_pct:+.2f}%)")
print(f"{'='*70}")

## Step 8: Visualize Forecast

In [ ]:
# Combine historical and forecast
historical_dates = data.index[-30:]
historical_prices = data['Close'].iloc[-30:].values

fig, ax = plt.subplots(figsize=(14, 7))

# Historical data
ax.plot(historical_dates, historical_prices, 
       label='Historical Price', linewidth=2.5, color='blue', alpha=0.8)

# Forecast
forecast_dates = pd.to_datetime(future_dates)
all_dates = historical_dates.tolist() + forecast_dates.tolist()
all_prices = list(historical_prices) + list(predictions_actual.flatten())

ax.plot(forecast_dates, predictions_actual.flatten(), 
       label='Forecasted Price', linewidth=2.5, color='red', alpha=0.8, linestyle='--')

# Confidence interval (simple)
std_error = np.std(y_actual.flatten() - y_pred_actual.flatten())
ax.fill_between(forecast_dates, 
               predictions_actual.flatten() - 1.96*std_error,
               predictions_actual.flatten() + 1.96*std_error,
               alpha=0.2, color='red', label='95% Confidence Interval')

# Styling
ax.set_title(f'{ticker} - {days_ahead}-Day Price Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Stock Price ($)', fontsize=12)
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 9: Comparison with Different Horizons

In [ ]:
# Compare predictions at different horizons
horizons = [7, 15, 30]
predictions_dict = {}

print(f"\nPredictions at Different Time Horizons:")
print(f"{'='*70}")

for horizon in horizons:
    if horizon <= len(predictions_actual):
        pred_price = predictions_actual[horizon-1, 0]
        change = pred_price - last_actual_price[0, 0]
        change_pct = (change / last_actual_price[0, 0]) * 100
        
        print(f"\nDay {horizon}:")
        print(f"  Predicted Price: ${pred_price:.2f}")
        print(f"  Change from Current: ${change:+.2f} ({change_pct:+.2f}%)")
        
        predictions_dict[horizon] = pred_price

print(f"\n{'='*70}")

## Summary

Key findings from model evaluation:
- Model successfully loaded and evaluated
- Recent performance metrics calculated
- 30-day forecast generated
- Trend analysis completed

### Important Notes:
- **This is for educational purposes only**
- Predictions are based on historical patterns
- Market conditions change; past performance ≠ future results
- Do not use for actual trading without professional advice
- Always conduct thorough risk analysis